# Political Violence in Myanmar (ACLED Analysis)
This notebook cleans and explores ACLED data to examine how political violence changed after the 2021 military coup. Before this, the study used the ACLED data export tools to filter down results, and create a more managable data set.

## Data Selection (ACLED Export Filters)

The dataset was downloaded from the ACLED Data Export Tool using the following filters:

- Country: Myanmar  
- Date range: 1 December 2018 to 4 May 2025  

### Event Types Included
- Battles  
- Explosions/Remote violence  
- Violence against civilians  

### Actors Included
- State forces  
- Rebel group  
- Political militia  
- Identity militia  
- Rioters  
- Protesters  
- Civilians  

### Additional Settings
- External/Other forces included  

These filters were selected to focus on direct forms of political violence while capturing interactions between state actors, armed groups, and civilians. Demonstration events were excluded in order to focus on violent interactions rather than protest activity.

# 1 Loading the Dataset
The next section begins the preperation to load the dataset properly.

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Code" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "Data"

df = pd.read_csv(DATA_DIR / "ACLED Data.csv")
df.head()

,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,...,location,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp
0,MMR6789,2019-04-29,2019,1,Political violence,Battles,Armed clash,KNU/KNLA: Karen National Union/Karen National ...,NaN,Rebel group,...,Kyainseikgyi,16.0408,98.1232,1,Irrawaddy,National,"On 29 April 2019, near the area of Three-Pagod...",0,NaN,1557838713
1,MMR7145,2019-02-26,2019,1,Political violence,Violence against civilians,Attack,Private Security Forces (Myanmar),Labor Group (Myanmar),External/Other forces,...,Waingmaw,25.3502,97.4384,1,Irrawaddy,National,"On 26 February 2019, in Waingmaw Township, in ...",0,NaN,1559057912
2,MMR6387,2019-01-08,2019,2,Political violence,Violence against civilians,Abduction/forced disappearance,PSLF/TNLA: Palaung State Liberation Front/Ta'a...,NaN,Rebel group,...,Namhkan,23.8334,97.6798,2,Myanmar Times,National,"On 8 January 2019, in Namkham Township, in Sha...",0,NaN,1618564830
3,MMR6478,2019-02-16,2019,2,Political violence,Violence against civilians,Attack,Unidentified Armed Group (Myanmar),NaN,Political militia,...,Maungdaw,20.8265,92.3661,2,Myanmar Times,National,"On 16 February 2019, near Taung Pyo Let Wae Vi...",3,NaN,1563908691
4,MMR6242,2018-12-16,2018,1,Political violence,Violence against civilians,Attack,Unidentified Armed Group (Myanmar),NaN,Political militia,...,Maungdaw,20.8265,92.3661,2,Irrawaddy,National,"On 16 December 2018, in Maungdaw township, Rak...",1,NaN,1563908691


## Convert Event Dates

In [2]:
df['event_date'] = pd.to_datetime(df['event_date'])

## Standardize Event Type Labels

Event type labels are cleaned to make later graphs and report outputs easier to read.

In [3]:
# Define the day of the coup
coup_date = pd.to_datetime("2021-02-01")

# Create period variable
df['period'] = df['event_date'].apply(
    lambda x: 'Before Coup' if x < coup_date else 'After Coup'
)

# Create month variable
df['month'] = df['event_date'].dt.to_period('M')

# Clean fatalities
df['fatalities'] = df['fatalities'].fillna(0)

df[['event_date', 'period', 'fatalities']].head()

,event_date,period,fatalities
0,2019-04-29,Before Coup,0
1,2019-02-26,Before Coup,0
2,2019-01-08,Before Coup,0
3,2019-02-16,Before Coup,3
4,2018-12-16,Before Coup,1


## Standardize Event Type Labels

Event type labels are cleaned to make later graphs and report outputs easier to read.

In [4]:
df['event_type'] = df['event_type'].replace({
    'Explosions/Remote violence': 'Explosions and Remote Attacks',
    'Violence against civilians': 'Violence Against Civilians'
})

df['event_type'].unique()

<StringArray>
['Battles', 'Violence Against Civilians', 'Explosions and Remote Attacks']
Length: 3, dtype: str

## Save Cleaned Dataset

The cleaned dataset is saved as a CSV file so it can be reused in the analysis notebook and report generation script.

In [5]:
df.to_csv(DATA_DIR / "cleaned_acled.csv", index=False)